# Lab 5 — Kalman Filtering and the Stochastic Cart

**ROB7104 / ROB8104 Mathematics for Robotics · the Kalman filter, EKF and UKF, and the banana distribution**

This notebook is a reading. There is no code to run and nothing to hand in. Every idea comes with a short video of a simulated robot, and every video has a live version on the lab's web page where you can turn the camera and move the sliders yourself:

**Interactive page:** [realgaule.github.io/rob7104/lab5](https://realgaule.github.io/rob7104/lab5/)

It follows Lecture 10 (discretising an SDE, the Kalman filter, Kalman–Bucy), Lecture 11+12 (the stochastic cart and exponential coordinates) and Lecture 14 (the Bayes filter and robot localisation). The equations of the EKF and the UKF are not on the slides; they are added here following Thrun, Burgard and Fox, *Probabilistic Robotics*, chapters 3 and 7, and Julier and Uhlmann (2004).

> If a video does not play inside your notebook viewer, open `Lab5.html` (same folder) in a browser.

## 0. Where is the cart?

<video controls loop muted playsinline width="100%" src="media/s00_opening.mp4" poster="media/s00_opening_poster.png"></video>

*2000 copies of one cart receive the same command, "drive 1 m straight ahead", with noisy wheels. Their end points spread into a curved cloud, the **banana**. The yellow ellipse is the $2\sigma$ ellipse of a Gaussian fitted to the end points in $(x, y)$: it has the wrong shape. Then two beacons send range readings (green rings), and the cloud is cut down to the carts that agree with them.*

**One idea carries the whole lab: what a robot knows about its own position is a probability distribution. A filter is a short way of writing that distribution down.**

Every filter repeats two steps (Lecture 14):

$$
\textbf{predict:}\quad p(x_k \mid z_{1:k-1}) = \int p(x_k \mid x_{k-1})\, p(x_{k-1} \mid z_{1:k-1})\, dx_{k-1},
\qquad
\textbf{correct:}\quad p(x_k \mid z_{1:k}) \;\propto\; p(z_k \mid x_k)\; p(x_k \mid z_{1:k-1}).
$$

Predicting pushes the distribution through the motion model, and it gets **wider**. Correcting conditions it on a measurement $z_k$, and it gets **narrower**. The video shows both. Driving without readings, the cloud grows to an rms spread of 0.34 m. Two range readings shrink it to 0.04 m. One range reading alone is not enough: it leaves the carts on a ring, and in the run shown the ring cuts the banana in two places, so the spread is even larger than before (0.39 m). On average over many runs a reading never increases the spread, but a single reading can.

The fitted Gaussian holds about the right share of the carts (89% inside its $2\sigma$ ellipse, where a Gaussian would hold 86%). What it gets wrong is the **shape**: about a tenth of the area inside the ellipse holds no carts at all, while the two horns of the banana stick out of it.

| Part | The distribution | The filter |
|---|---|---|
| **1. Linear Kalman filter** | stays Gaussian | mean and covariance lose nothing |
| **2. The stochastic cart** | bends into a banana | a Gaussian in $(x, y, \theta)$ has the wrong shape; one in exponential coordinates comes close |
| **3. Nonlinear filters** | banana, then banana cut by a ring | EKF, UKF; and the particle filter when no Gaussian fits |

**Links to earlier labs.** The blue cloud of carts is the particle picture of an SDE from Lab 3; Lab 3 only predicted, this lab adds measurements. In Part 1 the cart measures only its position and still finds its velocity because the pair $(F, H)$ is observable (Lab 4, 3.1).

**A word on what you are watching.** The carts move kinematically: the wheel speeds are the commanded ones plus noise, and the pose follows the SDE of the lecture. It is not a simulation of friction, slip or contact. MuJoCo only draws the scene. Colours keep one meaning throughout: the solid **white-topped cart** is the truth (the filter never sees it), the **blue** cloud is the distribution (many carts, or the particles of a particle filter), **yellow** is a Gaussian filter (EKF in Part 3), **orange** the UKF, **green** a reading, **pink** exponential coordinates, **red** a failure or "no readings".

> **Takeaway.** Predicting widens the belief, correcting narrows it; the filter is only a summary of it.
> ▶ Try it: [Where is the cart?](https://realgaule.github.io/rob7104/lab5/#d00)

---

# Part 1 — The linear Kalman filter

A cart drives along a straight track. A sensor reads its position, with an error of standard deviation 1 m, every $\Delta t = 0.5$ s. Its velocity is never measured. This is the example of Lecture 10.

## 1.1 Predict and correct

**The model.** Position $p$ and velocity $v$ obey the SDE $dp = v\,dt$, $dv = \sqrt{q}\,dW$: the speed wanders like a Brownian motion. Sampled every $\Delta t$, with state $x = (p, v)^\top$, it becomes exactly

$$
x_k = F x_{k-1} + w_k,\quad
F = \begin{pmatrix}1 & \Delta t\\ 0 & 1\end{pmatrix},\quad
w_k \sim N(0, Q),\quad
Q = q\begin{pmatrix}\Delta t^3/3 & \Delta t^2/2\\ \Delta t^2/2 & \Delta t\end{pmatrix},
\qquad
z_k = H x_k + n_k,\quad H = \begin{pmatrix}1 & 0\end{pmatrix},\quad n_k \sim N(0, R).
$$

The lecture uses $q = 0.05$ and $R = 1\ \text{m}^2$.

**The filter.** The belief is a Gaussian with mean $\hat x$ and covariance $P$. A linear map of a Gaussian is Gaussian, and a Gaussian times a Gaussian likelihood is Gaussian, so the two steps of Section 0 reduce to formulas for $\hat x$ and $P$:

$$
\textbf{predict:}\quad \hat x^- = F\hat x^+,\qquad P^- = FP^+F^\top + Q
$$

$$
\textbf{correct:}\quad \nu = z - H\hat x^-,\quad S = HP^-H^\top + R,\quad K = P^-H^\top S^{-1},\quad \hat x^+ = \hat x^- + K\nu,\quad P^+ = (I - KH)\,P^- .
$$

$\nu$ is the **innovation** (how much the reading surprises the filter) and $K$ the **gain**. In one dimension the correction is a weighted average. With a prior at 10 m with variance 4 and a reading of 12 m with variance 1: $K = 4/(4+1) = 0.8$, the estimate is $10 + 0.8\cdot 2 = 11.6$ m, and the variance is $0.8\ \text{m}^2$. The precisions add: $1/0.8 = 1/4 + 1/1$.

<video controls loop muted playsinline width="100%" src="media/s11_tunnel.mp4" poster="media/s11_tunnel_poster.png"></video>

*The cart passes through a tunnel where the sensor gets no readings, then comes out. Yellow: the filter's estimate and its $2\sigma$ ellipse; blue: the distribution itself, drawn as carts. Right: the belief in the $(p, v)$ plane, and below it prior $\times$ reading $\propto$ posterior at the first reading after the tunnel. The scene is the lecture example scaled down by 10 (the sensor error is 10 cm).*

In the tunnel the filter can only predict. Each prediction shears the ellipse (the term $F P F^\top$ moves position by velocity times time) and inflates it (the term $Q$). After 8 s, which is 15 missed readings, the position variance has grown from $0.34$ to $18.34\ \text{m}^2$ (standard deviation from 0.58 m to 4.28 m). It adds up from four sources:

$$
P_{pp}(T) = \underbrace{P_{pp}}_{0.34} + \underbrace{2T\,P_{pv}}_{2.11} + \underbrace{T^2 P_{vv}}_{7.36} + \underbrace{qT^3/3}_{8.53}\qquad (T = 8\ \text{s}).
$$

The velocity uncertainty and the noise account for most of it. At the exit the first reading arrives. The gain is $K = (0.948,\ 0.137)$: the filter now trusts the reading almost completely, and the position standard deviation drops from 4.28 m to 0.97 m in one step. After 13 more readings it is back at its steady value. Throughout, the filter stays honest: over 20000 runs, 95.45% of true carts lie inside its $2\sigma$ interval at the exit, which is the Gaussian value.

The blue cloud is the same linear model run with many random noises. It fills the yellow ellipse exactly as a Gaussian should: on average 86% of it lies inside the $2\sigma$ ellipse, and the Gaussian value is 86.5%. **For a linear model with Gaussian noise the mean and covariance are not an approximation: they are the whole distribution.**

> **Takeaway.** Without readings the ellipse shears and grows; one reading multiplies prior by likelihood and shrinks it.
> ▶ Try it: [Predict and correct](https://realgaule.github.io/rob7104/lab5/#d11): step the filter by hand.

## 1.2 The gain: trust the model or the sensor

**Question.** The filter is told $Q$ (how much the cart wanders) and $R$ (how bad the sensor is). What do they decide?

The gain. For this model,

$$
K = \begin{pmatrix}K_p\\ K_v\end{pmatrix} = \frac{1}{P^-_{pp} + R}\begin{pmatrix}P^-_{pp}\\ P^-_{pv}\end{pmatrix}.
$$

$K_p$ says which share of the innovation goes into the position. A large $Q$ makes the prior wide and $K_p$ close to 1: the filter follows the readings. A small $Q$ makes $K_p$ small: the filter follows its model. With the lecture's values the gain settles within about 7 s to $K = (0.33,\ 0.13\ \text{s}^{-1})$, with $\sigma_p = 0.57$ m and $\sigma_v = 0.34$ m/s (Lecture 10).

**How a position reading corrects the velocity.** $K_v$ is proportional to $P^-_{pv}$, the correlation between position and velocity. Prediction creates it: a cart that is ahead of the estimate is probably also faster, so the ellipse in the $(p, v)$ plane tilts. A reading that says "ahead" then also says "faster".

<video controls loop muted playsinline width="100%" src="media/s12a_gain.mp4" poster="media/s12a_gain_poster.png"></video>

*One drive with speed changes, and one sequence of position readings, fed to three filters that differ only in the $Q$ they assume: 100 times too large (trusts the sensor, $K_p = 0.72$), correct (tuned, $K_p = 0.33$), 100 times too small (trusts the model, $K_p = 0.12$). White arrows: true velocity; yellow arrows: estimated velocity. Right: the tilted ellipse of the tuned filter and how each gain splits the innovation. Lecture example scaled down by 10.*

| Filter (scaled scene) | Claimed $\sigma_p$ | Actual rms error | Truth outside the claimed $2\sigma$ |
|---|---|---|---|
| trusts the sensor ($Q \times 100$) | 8.5 cm | 7.9 cm | 3% of readings |
| tuned | 5.7 cm | 5.7 cm | 4% |
| trusts the model ($Q \times 0.01$) | 3.4 cm | 16.9 cm | 61% |

The bare sensor has an error of 10 cm. The jittery filter beats it a little, the tuned one clearly. The smooth one is worse than the sensor and does not know it: it claims 3.4 cm and is off by 17 cm (rms). In all three, the velocity arrow tracks the true velocity although velocity is never measured.

**Mistuning is not symmetric** (lecture scale, 2000 runs). With $Q$ 100 times too small the position error is 1.69 m, worse than the 1.0 m sensor, while the filter claims 0.34 m. With $Q$ 100 times too large it is 0.79 m, still better than the sensor. The innovations give a test that needs no ground truth: their normalised square $\nu^2/S$ (the NIS) has mean 1 when the filter is right. Averaged over 20 s windows it flags $Q \times 0.01$ in 89% of the windows but $Q \times 10$ in only 11% (the tuned filter: 5%). A filter that is too confident is easy to catch. A filter that is too cautious is hard to catch, but it also does less harm.

**The continuous-time limit.** Let the readings come faster, $\Delta t \to 0$, with $R = R_c/\Delta t$ so that the information per second stays fixed. The sawtooth of $P$ (up when predicting, down when correcting) then closes onto the solution of the Riccati equation $\dot P = AP + PA^\top + Q_c - PH^\top R_c^{-1}HP$: the Kalman–Bucy filter of Lecture 10. Without the last term it is the covariance equation $\dot P = AP + PA^\top + Q_c$, whose solution from $P = 0$ is the Gramian of Lab 4 (3.3) with $BB^\top = Q_c$: where noise can push the state.

> **Takeaway.** $Q$ against $R$ sets the gain; the tilt of the ellipse lets a position reading correct the velocity.
> ▶ Try it: [The gain](https://realgaule.github.io/rob7104/lab5/#d12): change $Q$ and $R$ and push the cart with the keyboard.

---

# Part 2 — The stochastic cart and the banana distribution

Now the cart drives on a floor, so its pose has three numbers, $(x, y, \theta)$, and the heading enters through $\cos\theta$ and $\sin\theta$. The model is no longer linear.

## 2.1 A cart with noisy wheels

**The model** (Lecture 11+12; Zhou and Chirikjian 2003; Long et al. 2012). A differential-drive cart has wheel radius $r = 0.033$ m and axle length $l = 0.2$ m. Its wheels turn at the commanded speeds $\omega_1, \omega_2$ plus white noise of strength $D$:

$$
\begin{aligned}
dx &= \tfrac{r}{2}(\omega_1 + \omega_2)\cos\theta\,dt + \sqrt{D}\,\tfrac{r}{2}\cos\theta\,(dW_1 + dW_2),\\
dy &= \tfrac{r}{2}(\omega_1 + \omega_2)\sin\theta\,dt + \sqrt{D}\,\tfrac{r}{2}\sin\theta\,(dW_1 + dW_2),\\
d\theta &= \tfrac{r}{l}(\omega_1 - \omega_2)\,dt + \sqrt{D}\,\tfrac{r}{l}\,(dW_1 - dW_2).
\end{aligned}
$$

The forward speed is $v = \tfrac{r}{2}(\omega_1 + \omega_2)$ and the turn rate $\omega = \tfrac{r}{l}(\omega_1 - \omega_2)$. For a path of given length only the product $DT$ of noise strength and duration matters, so the settings below are named by $DT$.

**What the noise does.** On a straight run the heading error is a Brownian motion: its variance grows linearly,

$$
\sigma_\theta^2(t) = \frac{2Dr^2}{l^2}\,t .
$$

A cart whose heading is off by $\theta$ moves forward only by the share $\cos\theta$ of each step. For a Gaussian heading error $E[\cos\theta] = e^{-\sigma_\theta^2/2} < 1$, so **the average cart falls behind the command**:

$$
E[\text{forward distance}] = \int_0^T v\, e^{-\sigma_\theta^2(t)/2}\,dt .
$$

<video controls loop muted playsinline width="100%" src="media/s21_cart.mp4" poster="media/s21_cart_poster.png"></video>

*The command "1 m straight ahead in 1 s" ($DT = 7$) run once, seven times, then 2000 times, with glowing trails; then a quarter arc with 1 m of path (radius 0.64 m). Right: heading errors with their $\pm 2\sigma_\theta$ band; their variance against the straight line of theory; and how far the mean position falls behind the command, for $DT = 1, 4, 7$.*

| $DT$ | Heading sd at the end | Mean forward distance (of 1 m), theory | 40000 simulated carts |
|---|---|---|---|
| 1 | 0.23 rad (13°) | 0.9865 m | 0.9865 m |
| 4 | 0.47 rad (27°) | 0.9475 m | 0.9475 m |
| 7 | 0.62 rad (35°) | 0.9105 m | 0.9104 m |

At $DT = 7$ the cloud is a banana: the carts that turned left end up on the left and short, those that turned right on the right and short. On the quarter arc the banana is slanted, and the mean falls 8.3 cm behind the commanded end.

> **Takeaway.** Wheel noise makes the heading diffuse; the heading bends the cloud into a banana, and its mean falls behind the command.
> ▶ Try it: [The cart with noisy wheels](https://realgaule.github.io/rob7104/lab5/#d21)

## 2.2 Exponential coordinates: the banana nearly straightens

**Question.** The banana is not a Gaussian in $(x, y, \theta)$. Is it a Gaussian in some other coordinates?

**The idea** (Lecture 11+12; Long et al. 2012). Describe a pose not by where the cart stands but by **the constant twist that would take it there**: drive with forward speed $v_1$, sideways speed $v_2$ and turn rate $\alpha$ for one unit of time. These $\xi = (v_1, v_2, \alpha)$ are the exponential coordinates of the pose:

$$
\exp(\xi) = (x, y, \theta),\qquad
\begin{pmatrix}x\\ y\end{pmatrix} = \frac{1}{\alpha}\begin{pmatrix}\sin\alpha & -(1-\cos\alpha)\\ 1-\cos\alpha & \sin\alpha\end{pmatrix}\begin{pmatrix}v_1\\ v_2\end{pmatrix},\qquad \theta = \alpha .
$$

Carts that drove circular arcs of the same length $L$ but different curvature $\kappa$, which is roughly what a heading error does, lie on a curve in $(x, y)$ but on a straight line in $\xi$: $\xi = (L,\ 0,\ \kappa L)$. The model for the cloud is a Gaussian on the group of poses: $g = \mu \circ \exp(\xi)$ with $\xi \sim N(0, \Sigma)$.

**No samples needed.** To first order in the noise, the mean and covariance follow in closed form from the command $h = (v, 0, \omega)$ and the noise $HH^\top = D\,\mathrm{diag}(r^2/2,\ 0,\ 2r^2/l^2)$ (this $H$ maps the wheel noise into the twist; it is not the measurement matrix) (Lecture 11+12):

$$
\mu(t) = \exp(t\,h),\qquad
\Sigma(t) = \int_0^t \mathrm{Ad}_{\exp(-s h)}\, HH^\top\, \mathrm{Ad}_{\exp(-s h)}^\top\, ds .
$$

For a straight run, with $q_1 = Dr^2/2$ and $q_3 = 2Dr^2/l^2$,

$$
\Sigma(t) = \begin{pmatrix} q_1 t & 0 & 0\\ 0 & q_3 v^2 t^3/3 & q_3 v t^2/2\\ 0 & q_3 v t^2/2 & q_3 t\end{pmatrix}.
$$

<video controls loop muted playsinline width="100%" src="media/s23_expcoords.mp4" poster="media/s23_expcoords_poster.png"></video>

*The carts after 1 m straight ahead ($DT = 7$), lifted into the space $(x, y, \theta)$, where they form a curved sheet; then the space morphs into $(v_1, v_2, \alpha)$, where the sheet becomes nearly flat and a pink ellipsoid fits it. Mapped back to the floor, the pink Gaussian has banana-shaped contours. Last, the banana drawn from the closed-form $\mu(t)$ and $\Sigma(t)$ alone, growing with time. Right: the distance between each model and the cloud.*

How close is "nearly"? The table gives the total variation distance between each model and the true cloud on the floor (0 = identical, 1 = no overlap; two independent clouds of 200000 carts differ by about 0.03 from sampling alone):

| $DT$ (straight) | Gaussian in $(x, y, \theta)$ | Gaussian in $(v_1, v_2, \alpha)$, fitted to the carts | closed form $\mu(t)$, $\Sigma(t)$ | sampling floor |
|---|---|---|---|---|
| 1 | 0.168 | 0.067 | 0.058 | 0.030 |
| 4 | 0.295 | 0.149 | 0.108 | 0.025 |
| 7 | 0.350 | 0.202 | 0.142 | 0.027 |

Three things stand out.

- Exponential coordinates are always much closer, and the closer the less noise. They are **close to** the banana, not equal to it: at $DT = 7$ a bend remains (the cloud in the video is a warped sheet, not a perfect ellipsoid; its skewness drops from 5.1 to 2.6, where a Gaussian has 0). The paper's title says the banana "is Gaussian"; in exponential coordinates it nearly is, and the error grows with the noise without a sharp threshold.
- The closed form, which uses no samples at all, beats the Gaussian fitted to 200000 samples. The fitted Gaussian is widened by the leftover bend (its along-track variance is 0.0067 against 0.0038 in the closed form); the closed form ignores the bend and is instead a little overconfident in the tails.
- The closed form is a first-order result, and one entry shows it: the along-track variance $\Sigma_{11}$ is too small (0.0038 against 0.0067 measured at $DT = 7$). A second-order term, $\Sigma_{11} \approx q_1 T + \sigma_\theta^4 L^2/48$ for a path of length $L$, fixes it to within 2.5% up to $DT = 16$. The same expansion in Cartesian $x$ gives $\sigma_\theta^4 L^2/12$, four times more: exponential coordinates halve the bend, they do not remove it. The closed-form 95% ellipsoid holds 94.2% of the carts at $DT = 1$ and 90.1% at $DT = 7$.

The mean behaves differently in the two descriptions. The average $(x, y)$ of the carts is 9 cm short of the commanded end; the mean on the group, $\mu(T)$, stays at the commanded end.

> **Takeaway.** Choose coordinates that follow the motion and the banana becomes close to Gaussian; with little noise, its mean and covariance follow from the command alone.
> ▶ Try it: [Exponential coordinates](https://realgaule.github.io/rob7104/lab5/#d22): morph the cloud between the two coordinate systems.

---

# Part 3 — Nonlinear filters

The motion of the cart is $x_k = f(x_{k-1}, u_k, n_k)$, with the command $u_k = (v, \omega)$ and the wheel noise $n_k = (n_1, n_2) \sim N(0, D\,\Delta t\, I)$. One step of the filter model is

$$
ds = v\,\Delta t + \tfrac{r}{2}(n_1 + n_2),\qquad d\theta = \omega\,\Delta t + \tfrac{r}{l}(n_1 - n_2),\qquad
f(x, u, n) = \begin{pmatrix} x + ds\,\cos(\theta + d\theta/2)\\ y + ds\,\sin(\theta + d\theta/2)\\ \theta + d\theta\end{pmatrix}.
$$

A beacon at $b$ gives a range reading $z = \lVert p - b\rVert + m$, $m \sim N(0, \sigma_r^2)$, where $p = (x, y)$; some scenes also read the bearing of the beacon. Both $f$ and the reading are nonlinear, so the Gaussian does not stay Gaussian. The Kalman formulas need a mean and a covariance after each step; the filters below differ in how they get them.

## 3.1 EKF: linearise at the mean

The **extended Kalman filter** replaces $f$ and $h$ by their tangent at the current estimate (Thrun et al., ch. 3):

$$
\textbf{predict:}\quad \hat x^- = f(\hat x^+, u, 0),\qquad P^- = F P^+ F^\top + G\,(D\,\Delta t\,I)\,G^\top,\qquad F = \frac{\partial f}{\partial x}\Big|_{\hat x^+},\quad G = \frac{\partial f}{\partial n}\Big|_{\hat x^+}
$$

$$
\textbf{correct:}\quad H = \frac{\partial h}{\partial x}\Big|_{\hat x^-},\quad \nu = z - h(\hat x^-),\quad S = HP^-H^\top + R,\quad K = P^-H^\top S^{-1},\quad \hat x^+ = \hat x^- + K\nu,\quad P^+ = (I-KH)P^- .
$$

The Kalman formulas are unchanged; $F$ and $H$ are now Jacobians, evaluated at the estimate. For the range reading, $H = \big((p - b)^\top/\lVert p - b\rVert,\ 0\big)$: the reading measures only the distance to the beacon, but through the correlations in $P^-$ (as in 1.2) the correction $K\nu$ also moves the other components, the heading included.

<video controls loop muted playsinline width="100%" src="media/s31_ekf.mp4" poster="media/s31_ekf_poster.png"></video>

*First, the cart drives 1 m with a range-and-bearing reading every 0.5 s: the EKF's yellow estimate and ellipse follow it. Then the same command without readings ($DT = 7$): the EKF's estimate ends exactly on the commanded end mark, while the centre of the cloud of possible carts is 0.09 m behind. Right: the belief about the heading, pushed through $\cos\theta$ along the tangent (EKF) and along the curve (exact).*

With readings the EKF works well here: over 4000 runs its 95% ellipse holds the true cart in 94% of cases at the end, and no run loses the cart.

Without readings the linearisation shows. The EKF moves its mean with the noise-free model at the mean heading $\theta = 0$, forward by the full $v\,\Delta t$ at every step, as if the heading were known: through the tangent line, the mean heading maps to $\cos 0 = 1$. The true forward share is $E[\cos\theta] = 0.83$ at the end. This is the lag of 2.1, which the EKF cannot see. Its ellipse also cannot follow the banana: the "95%" ellipse actually holds

| $DT$ | 1 | 4 | 7 |
|---|---|---|---|
| true carts inside the EKF's 95% ellipse | 89% | 80% | 75% |

A remark that connects to 2.2: on this straight run the EKF's covariance is, to three digits, the closed-form $\Sigma(t)$ of exponential coordinates. The two methods carry the same numbers. The EKF reads them as an ellipse in $(x, y, \theta)$; exponential coordinates read them as a Gaussian on the group, which is a banana on the floor.

> **Takeaway.** The EKF is the Kalman filter with Jacobians; through a curved model its mean sits where the noise-free cart would be, not where the carts are.
> ▶ Try it: [EKF](https://realgaule.github.io/rob7104/lab5/#d31): drive blind for longer and watch the offset grow.

## 3.2 UKF: let a few points travel for the distribution

**Idea** (Julier and Uhlmann 2004). Instead of linearising the model, choose a few points that have the mean and covariance of the belief, push each one through the exact nonlinear model, and compute the mean and covariance of the images. This is the **unscented transform**.

For a state of dimension $n$ with mean $\bar x$ and covariance $P$, the $2n + 1$ **sigma points** and their weights are

$$
\mathcal X_0 = \bar x,\qquad \mathcal X_{\pm i} = \bar x \pm \Big(\sqrt{(n+\lambda)\,P}\Big)_i\ \ (i = 1, \dots, n),\qquad
W_0 = \frac{\lambda}{n+\lambda},\qquad W_{\pm i} = \frac{1}{2(n+\lambda)},
$$

where $(\cdot)_i$ is the $i$-th column of a matrix square root (Cholesky) and $\lambda = \alpha^2(n + \kappa) - n$. (Here $\alpha$ and $\kappa$ are the spread parameters of the transform, not the turn angle of 2.2, and $n$ is the dimension of the state.) The prediction is

$$
\mathcal Y_j = f(\mathcal X_j),\qquad \hat x^- = \sum_j W_j\,\mathcal Y_j,\qquad P^- = \sum_j W_j\,(\mathcal Y_j - \hat x^-)(\mathcal Y_j - \hat x^-)^\top .
$$

The correction draws new sigma points from $\hat x^-$ and $P^-$ and pushes them through $h$, giving a predicted reading $\hat z$, its covariance $P_{zz}$ (plus $R$) and the cross-covariance $P_{xz}$; then $K = P_{xz}P_{zz}^{-1}$ and the Kalman update as before (Thrun et al., ch. 3). Here the state $(x, y, \theta)$ is augmented with the two wheel noises, so $n = 5$ and there are 11 sigma points; with $\alpha = 1$, $\kappa = 1$ ($\lambda = 1$) the centre point has weight $1/6$ and the others $1/12$ (in general the centre's weight in the covariance gets an extra $1 - \alpha^2 + \beta$, which is 0 here with $\beta = 0$). No Jacobian is needed.

<video controls loop muted playsinline width="100%" src="media/s32_ukf.mp4" poster="media/s32_ukf_poster.png"></video>

*Driving blind, straight 1 m, $DT = 7$. At each step: (1) pick 11 sigma points from the estimate (7 of them stand at its centre), drawn as orange carts; (2) every point drives the same command; (3) their weighted average is the new estimate. Right: the same idea with the heading alone, three points with weights $\tfrac12, \tfrac14, \tfrac14$ pushed through $\cos\theta$.*

In the one-dimensional picture at the end of the run, $\sigma_\theta = 0.62$ rad and the points sit at $0$ and $\pm\sqrt{2}\,\sigma_\theta$. Their images under $\cos$ average to 0.821; the exact $E[\cos\theta]$ is 0.826, and the tangent (EKF) gives 1. Two side points are enough to feel the curvature.

| $DT$ | true mean forward distance | EKF | UKF | inside the 95% ellipse: EKF / UKF |
|---|---|---|---|---|
| 1 | 0.9865 m | 1.0000 m | 0.9867 m | 89% / 92% |
| 4 | 0.9475 m | 1.0000 m | 0.9486 m | 80% / 88% |
| 7 | 0.9105 m | 1.0000 m | 0.9130 m | 75% / 85% |

The UKF recovers 97% to 99% of the lag. It is better, not exact: along the track its standard deviation at $DT = 7$ is 6.7 cm while the carts spread by 11.2 cm, so its 95% ellipse holds only 85% of them. A mean and a covariance cannot draw a banana, however well they are computed.

> **Takeaway.** The UKF sends a few points through the true model instead of a tangent; it gets the mean right, but it is still one ellipse.
> ▶ Try it: [UKF](https://realgaule.github.io/rob7104/lab5/#d32): switch between EKF and UKF and show the sigma points.

## 3.3 When a Gaussian is not enough

**The test.** The cart drives an arc (1 m/s, radius 1.27 m). For a while the only beacon sends nothing and the cart drives blind; then range readings ($\sigma_r = 3$ cm) arrive for 1 s. Three filters run on the same readings: EKF, UKF, and a **particle filter**, which keeps the cloud itself: each particle drives the noisy model, is weighted by the likelihood of the reading $p(z \mid x^{(i)})$, and the cloud is resampled by weight (Thrun et al., ch. 4; Lecture 14). A filter has **lost** the cart when its estimate is more than 0.15 m off and the cart lies outside the filter's own 99.99% ellipse (for the particle filter: farther from the cloud's mean than every particle).

<video controls loop muted playsinline width="100%" src="media/s33_beyond_gaussian.mp4" poster="media/s33_beyond_gaussian_poster.png"></video>

*First a mild setting (wheel noise $D = 1$, 0.5 s without readings): all three filters keep the cart. Then a severe one ($D = 8$, 1.5 s without readings): the blue particle cloud is the banana cut by the green range ring; the yellow EKF and orange UKF ellipses settle on the wrong part of the ring. The run shown is one where both are lost, which happens in 15% of runs. Right: top view, the outcome shares of the severe setting, and the share of lost runs over 25 settings.*

**What goes wrong.** After the blind stretch the belief is a banana. One range reading says "the cart is on this ring". The true posterior is the part of the banana that the ring cuts out: a thin curved arc, and sometimes two separate pieces (in the severe setting the second piece holds 5% of the probability). A mean and a covariance describe one ellipse. They cannot describe a curved arc, and still less two pieces. This is the limit that Lecture 10 names: the Kalman filter assumes linear models and Gaussian noise.

Share of runs that lose the cart at the end, **EKF / UKF**, in per cent (4000 runs per setting). Rows: wheel noise $D$; columns: time without readings.

| $D$ | 0.5 s | 1 s | 1.5 s | 2 s | 3 s |
|---|---|---|---|---|---|
| 0.5 | 0 / 0 | 0 / 0 | 0 / 0 | 1 / 0 | 25 / 10 |
| 1 | 0 / 0 | 0 / 0 | 1 / 1 | 3 / 1 | 43 / 20 |
| 2 | 0 / 0 | 3 / 1 | 6 / 4 | 11 / 5 | 64 / 29 |
| 4 | 3 / 1 | 9 / 4 | 18 / 10 | 26 / 13 | 75 / 37 |
| 8 | 8 / 4 | 20 / 10 | 35 / 18 | 48 / 26 | 83 / 49 |

The particle filter (3000 particles per run, drawn from a shared pool of 200000; 400 runs per setting) loses the cart in at most 2 of 400 runs in any setting, 7 of 10000 runs in all.

What the numbers say, and what they do not:

- **The two Gaussian filters fail together.** There is no noise level at which the EKF fails and the UKF still works: in none of 100 settings scanned (several beacon layouts) does that happen. The UKF fails later and loses about half as many runs (0.34 to 0.52 times as many). In the severe setting, 62% of runs end with both on track, 21% with only the EKF lost, 3% with only the UKF lost, and 15% with both lost; 83% of the UKF's losses are also EKF losses.
- **A better sensor makes them worse.** In the severe setting, a range error of 1 cm instead of 3 cm makes the EKF lose 46% of runs instead of 35% (UKF: 21% instead of 17%); 10 cm gives 30% (15%). A sharper ring cuts a thinner arc out of the banana, and the ellipse that has to cover it fails more often.
- **The particle filter is honest, not accurate.** With one beacon its end error is still 0.34 to 0.44 m (UKF 0.61 to 0.64 m, EKF 0.73 to 0.83 m), because one ring leaves the cart's place along the ring open. What it does is keep the true cart inside its cloud, so it knows how little it knows.
- **It needs enough particles.** At the first reading only about 6% of the particles carry real weight. With 1000 particles per run a plain particle filter loses the cart in 1% to 3.7% of runs; with 16000, in at most 0.3%. A live demo with about 1000 particles will show such failures.

> **Takeaway.** When the belief is a banana cut by a ring, any mean-plus-covariance filter can jump to the wrong place; only a filter that keeps the shape keeps the cart.
> ▶ Try it: [Beyond the Gaussian](https://realgaule.github.io/rob7104/lab5/#d33): drive the cart yourself and hide the beacon.

---

## 4. Summary

Every filter in this lab is the same loop, predict and correct, applied to a belief. They differ in how they write the belief down.

| Filter | How it writes down the belief | Cost per step | When it fails |
|---|---|---|---|
| **Kalman filter** | mean and covariance | a few small matrix products | never, for linear models with Gaussian noise and the right $Q$, $R$; a too-small $Q$ makes it overconfident |
| **EKF** | mean and covariance, models linearised at the mean | plus two Jacobians | curved models: mean at the noise-free point, ellipse too small (75% for "95%" at $DT = 7$) |
| **UKF** | mean and covariance, from $2n+1$ points through the true model | $2n+1$ model evaluations | mean nearly right, spread still too small (85%); fails with the EKF, later and about half as often |
| **Exponential coordinates** | a Gaussian on the group of poses | closed form, no samples | close to the banana, not equal; closer the less noise |
| **Particle filter** | the cloud itself | thousands of model evaluations | too few particles |

**What this lab left out.** Exponential coordinates were used here only to represent and predict the cart's distribution; filters that run in these coordinates (invariant or Lie-group EKFs) are the natural next step. Also not covered: smoothing (using later readings to improve earlier estimates), building the map while localising (SLAM), and the details of resampling.

**Notes on the simulations.**

- The carts move kinematically, following the SDE of 2.1 (Euler–Maruyama with a 1 ms step). There is no friction, slip or contact model; MuJoCo only renders the scene.
- The filters use a coarser step model (0.025 to 0.1 s per step) than the simulated truth, so they carry a small, realistic model error.
- The cart's dimensions and the noise settings follow Long et al. (2012); our simulated clouds agree with the data published with that paper.
- Every number in this notebook comes from the scripts that made the videos; the simulator and the filters were checked against an independent implementation and against `filterpy`. Shares of lost runs are rounded to whole per cent.

**References.** S. J. Julier and J. K. Uhlmann, "Unscented filtering and nonlinear estimation," *Proceedings of the IEEE* 92(3), 2004. S. Thrun, W. Burgard and D. Fox, *Probabilistic Robotics*, MIT Press, 2005 (chapters 3, 4, 7). Y. Zhou and G. S. Chirikjian, "Probabilistic models of dead-reckoning error in nonholonomic mobile robots," ICRA 2003. A. W. Long, K. C. Wolfe, M. J. Mashner and G. S. Chirikjian, "The banana distribution is Gaussian: a localization study with exponential coordinates," RSS 2012.